In [2]:
from training_utilities_2nd_part import *

In [3]:
# weather

from variables_to_specify_weather import *
df, columns_to_normalize, weather_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, weather_drop_columnss, weather_windows, index_of_one_month, one_month_window_size = variables_to_specify_weather()

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
df = df.dropna().reset_index(drop=True)
convert_time(df, 'Date Time')
weather_df = df
weather_time_steps = 1


# stationary

In [5]:
weather_len_of_training_data_of_stationary_model =15*No_of_datapoints_in_one_day

train = df[0:weather_len_of_training_data_of_stationary_model] 
test = df[weather_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 1, out_columns, weather_target_col, weather_drop_columnss)

sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)

print(eval_df_first_month['Testing Error'].mean())
print(eval_df_first_month['mae'].mean())

Model Type: RandomForestRegressor
Storage Required: 11.65 MB
model storage is : 11.647904396057129


total_time is:  1.6258859159999997
sum_training_time is:  14.031468330000003
0.01412362662340654
0.06907236825523469


# Model reuse

In [6]:
# Model reuse
daily_df_avg = get_elect_daily_avg(weather_df, No_of_datapoints_in_one_day, weather_target_col, avg_target_col_name)


seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 15)

Detected seasonality periods (ACF): [15 31 38]
median_value is:  31


# data drift detection

In [7]:
df_copy = weather_df[[weather_target_col]]
target_col = weather_target_col
time_steps = weather_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 15* multiplier
window_len_=[x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=weather_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: I

In [8]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "SA", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)


window is:  2160
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 11.65 MB


window is:  4320
i/window is :  2.0
similar_month_index is :  0
month_index:  2




window is:  6480
i/window is :  3.0
Model Type: RandomForestRegressor
Storage Required: 11.83 MB


window is:  8640
i/window is :  4.0
Model Type: RandomForestRegressor
Storage Required: 11.46 MB


window is:  10800
i/window is :  5.0
similar_month_index is :  2
previous_model_i is :  10800
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: RandomForestRegressor
Storage Required: 13.49 MB


window is:  12960
i/window is :  6.0
similar_month_index is :  3
month_index:  6




window is:  15120
i/window is :  7.0
Model Type: RandomForestRegressor
Storage Required: 14.23 MB


window is:  17280
i/window is :  8.0
similar_month_index is :  6
previous_model_i is :  17280
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: RandomForestRegressor
Storage Required: 12.

In [9]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "SA", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  2160
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 11.65 MB


window is:  4320
i/window is :  2.0
Model Type: RandomForestRegressor
Storage Required: 12.37 MB


window is:  6480
i/window is :  3.0
Model Type: RandomForestRegressor
Storage Required: 11.83 MB


window is:  8640
i/window is :  4.0
Model Type: RandomForestRegressor
Storage Required: 11.46 MB


window is:  10800
i/window is :  5.0
similar_month_index is :  1
month_index:  5




window is:  12960
i/window is :  6.0
similar_month_index is :  0
month_index:  6




window is:  15120
i/window is :  7.0
similar_month_index is :  0
month_index:  7




window is:  17280
i/window is :  8.0
similar_month_index is :  3
month_index:  8




window is:  19440
i/window is :  9.0
similar_month_index is :  7
previous_model_i is :  19440
math.floor(previous_model_i/window) is:  9
len(models_ls) is: 8
Model Type: RandomForestRegressor
Storage Required: 12.72 MB


window is:  21600
i/window is :  10.0
simil

In [10]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "ES", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  2160
Model Type: RandomForestRegressor
Storage Required: 11.65 MB


window is:  4320
Model Type: RandomForestRegressor
Storage Required: 12.37 MB


window is:  6480
similar_month_index is :  0
month_index:  2




window is:  8640
Model Type: RandomForestRegressor
Storage Required: 11.46 MB


window is:  10800
similar_month_index is :  2
previous_model_i is :  10800
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: RandomForestRegressor
Storage Required: 13.49 MB


window is:  12960
Model Type: RandomForestRegressor
Storage Required: 13.49 MB


window is:  15120
similar_month_index is :  0
month_index:  6




window is:  17280
similar_month_index is :  3
month_index:  7




window is:  19440
similar_month_index is :  1
month_index:  8




window is:  21600
similar_month_index is :  5
month_index:  9




window is:  23760
similar_month_index is :  8
previous_model_i is :  23760
math.floor(previous_model_i/window) is:  11
len(models_ls) is: 10
Model T

In [11]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "ES", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  2160
Model Type: RandomForestRegressor
Storage Required: 11.65 MB


window is:  4320
Model Type: RandomForestRegressor
Storage Required: 12.37 MB


window is:  6480
Model Type: RandomForestRegressor
Storage Required: 11.83 MB


window is:  8640
Model Type: RandomForestRegressor
Storage Required: 11.46 MB


window is:  10800
Model Type: RandomForestRegressor
Storage Required: 11.53 MB


window is:  12960
Model Type: RandomForestRegressor
Storage Required: 13.49 MB


window is:  15120
Model Type: RandomForestRegressor
Storage Required: 14.23 MB


window is:  17280
Model Type: RandomForestRegressor
Storage Required: 13.42 MB


window is:  19440
similar_month_index is :  2
month_index:  8




window is:  21600
similar_month_index is :  7
month_index:  9




window is:  23760
Model Type: RandomForestRegressor
Storage Required: 11.87 MB


window is:  25920
similar_month_index is :  8
previous_model_i is :  25920
math.floor(previous_model_i/window) is:  12
len(models_ls) is: 11
Mo

In [15]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

12.198133754730224


# informed

In [13]:
informed_update(stationary_model1,weather_df, target_col, weather_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 1)

window is:  2160
Model Type: RandomForestRegressor
Storage Required: 11.65 MB
window is:  4320
Model Type: RandomForestRegressor
Storage Required: 12.37 MB
window is:  6480
Model Type: RandomForestRegressor
Storage Required: 11.83 MB
window is:  8640
Model Type: RandomForestRegressor
Storage Required: 11.46 MB
window is:  10800
Model Type: RandomForestRegressor
Storage Required: 11.53 MB
window is:  12960
window is:  15120
Model Type: RandomForestRegressor
Storage Required: 14.23 MB
window is:  17280
Model Type: RandomForestRegressor
Storage Required: 13.42 MB
window is:  19440
Model Type: RandomForestRegressor
Storage Required: 12.98 MB
window is:  21600
window is:  23760
Model Type: RandomForestRegressor
Storage Required: 11.87 MB
window is:  25920
window is:  28080
window is:  30240
Model Type: RandomForestRegressor
Storage Required: 12.86 MB
window is:  32400
window is:  34560
Model Type: RandomForestRegressor
Storage Required: 12.06 MB
window is:  36720
Model Type: RandomForestReg

# periodical

In [14]:
periodical_retraining_with_hptuning(1, weather_df, weather_windows, out_columns, weather_target_col, weather_drop_columnss)

window size is :  720
Model Type: RandomForestRegressor
Storage Required: 4.84 MB
Model Type: RandomForestRegressor
Storage Required: 4.72 MB
Model Type: RandomForestRegressor
Storage Required: 4.70 MB
Model Type: RandomForestRegressor
Storage Required: 4.81 MB
Model Type: RandomForestRegressor
Storage Required: 4.95 MB
Model Type: RandomForestRegressor
Storage Required: 5.04 MB
Model Type: RandomForestRegressor
Storage Required: 4.82 MB
Model Type: RandomForestRegressor
Storage Required: 4.74 MB
Model Type: RandomForestRegressor
Storage Required: 4.69 MB
Model Type: RandomForestRegressor
Storage Required: 4.81 MB
Model Type: RandomForestRegressor
Storage Required: 4.95 MB
Model Type: RandomForestRegressor
Storage Required: 4.65 MB
Model Type: RandomForestRegressor
Storage Required: 5.01 MB
Model Type: RandomForestRegressor
Storage Required: 4.69 MB
Model Type: RandomForestRegressor
Storage Required: 5.03 MB
Model Type: RandomForestRegressor
Storage Required: 5.36 MB
Model Type: Random

([         Training dataset     Testing dataset       mae       mse      rmse  \
  0   trained on window i-1  tested on window i  0.008524  0.000314  0.017709   
  1   trained on window i-1  tested on window i  0.003271  0.000129  0.011346   
  2   trained on window i-1  tested on window i  0.020075  0.000588  0.024257   
  3   trained on window i-1  tested on window i  0.033706  0.002103  0.045863   
  4   trained on window i-1  tested on window i  0.012330  0.000535  0.023125   
  ..                    ...                 ...       ...       ...       ...   
  67  trained on window i-1  tested on window i  0.001499  0.000042  0.006449   
  68  trained on window i-1  tested on window i  0.011812  0.000469  0.021651   
  69  trained on window i-1  tested on window i  0.004315  0.000116  0.010757   
  70  trained on window i-1  tested on window i  0.009750  0.000365  0.019094   
  71  trained on window i-1  tested on window i  0.009209  0.000167  0.012939   
  
            r2      mape 